# Smart Room: XGBoost with Three Chronological Splits
Predict room occupancy from the same four sensor inputs used in the logistic regression experiment.

| Split | Dates in 2015 (inclusive) | Rows |
|---|---|---:|
| Train | February 2-11 | 11,360 |
| Validation | February 12-15 | 5,760 |
| Test | February 16-18 | 3,440 |

Target: `Occupancy`, where 0 = unoccupied and 1 = occupied. This notebook evaluates XGBoost only; no LR comparison, always-unoccupied baseline, or Macro F1 is included. The same splits, inputs, selection metric and prediction threshold support a later comparison. XGBoost is not assumed to outperform LR.

## 1. Set up the experiment
Keep this notebook and the three supplied CSV files together. Use a Python kernel with pandas, NumPy, scikit-learn and XGBoost installed.

If XGBoost is missing, run `%pip install xgboost` in a separate code cell, restart the kernel, and rerun from the top. No installation runs automatically in this notebook.

Features are Temperature, Humidity, Light and CO2. Date is used only for split checks, and HumidityRatio remains excluded. No source readings or labels are changed. Tree-based XGBoost uses feature split thresholds here.

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, balanced_accuracy_score,
                             confusion_matrix, classification_report, roc_auc_score)

DATA_DIR = Path('.')
FEATURES = ['Temperature', 'Humidity', 'Light', 'CO2']
TARGET = 'Occupancy'
SEED = 42

## 2. Load and verify the three files
Checks verify column structure, counts, dates, numerical validity, binary labels, and chronological separation. They do not verify labels against human observations. Validation and test data are checked for integrity but are not used to learn scaling or coefficients.

In [9]:
specs = {
    'Train': ('Occupancy_Train.csv', '2015-02-02', '2015-02-11', 11360),
    'Validation': ('Occupancy_Validation.csv', '2015-02-12', '2015-02-15', 5760),
    'Test': ('Occupancy_Test.csv', '2015-02-16', '2015-02-18', 3440),
}
expected_columns = ['date', 'Temperature', 'Humidity', 'Light',
                    'CO2', 'HumidityRatio', 'Occupancy']
datasets = {}
split_rows = []
for name, (filename, start, end, expected_rows) in specs.items():
    df = pd.read_csv(DATA_DIR / filename, parse_dates=['date'])
    assert list(df.columns) == expected_columns, f'{name}: unexpected columns'
    assert len(df) == expected_rows, f'{name}: unexpected row count'
    assert not df.isna().any().any(), f'{name}: missing cells'
    assert df['date'].is_monotonic_increasing, f'{name}: timestamps not sorted'
    assert not df['date'].duplicated().any(), f'{name}: repeated timestamps'
    assert df['date'].between(pd.Timestamp(start), pd.Timestamp(end) + pd.Timedelta(days=1), inclusive='left').all()
    assert df[TARGET].isin([0, 1]).all(), f'{name}: invalid target'
    assert set(df[TARGET].unique()) == {0, 1}, f'{name}: only one class'
    assert np.isfinite(df[FEATURES].to_numpy(dtype=float)).all()
    datasets[name] = df
    split_rows.append({'Split': name, 'Rows': len(df),
                       'First timestamp': str(df['date'].min()),
                       'Last timestamp': str(df['date'].max()),
                       'Unoccupied': int(df[TARGET].eq(0).sum()),
                       'Occupied': int(df[TARGET].eq(1).sum()),
                       'Occupied %': round(df[TARGET].mean() * 100, 2)})

assert datasets['Train']['date'].max() < datasets['Validation']['date'].min()
assert datasets['Validation']['date'].max() < datasets['Test']['date'].min()
assert pd.concat(datasets.values())['date'].nunique() == 20560
print(pd.DataFrame(split_rows).to_string(index=False))
print('Verified: chronological splits, no shared timestamps, all 20,560 rows accounted for.')

     Split  Rows     First timestamp      Last timestamp  Unoccupied  Occupied  Occupied %
     Train 11360 2015-02-02 14:19:00 2015-02-11 23:58:59        8445      2915       25.66
Validation  5760 2015-02-12 00:00:00 2015-02-15 23:58:59        5022       738       12.81
      Test  3440 2015-02-16 00:00:00 2015-02-18 09:19:00        2343      1097       31.89
Verified: chronological splits, no shared timestamps, all 20,560 rows accounted for.


## 3. Define features, target, and evaluation metrics
`X` contains the four sensor predictors and `y` the occupancy label. Precision, recall, and F1 refer to the occupied class (1). Accuracy alone can conceal weak detection of the minority occupied class. Balanced accuracy averages recall across both classes. ROC AUC evaluates how well predicted probabilities rank the classes.

In [10]:
X_train, y_train = datasets['Train'][FEATURES], datasets['Train'][TARGET]
X_val, y_val = datasets['Validation'][FEATURES], datasets['Validation'][TARGET]
X_test, y_test = datasets['Test'][FEATURES], datasets['Test'][TARGET]

def evaluate(y_true, predictions, probabilities=None):
    result = {
        'Accuracy': accuracy_score(y_true, predictions),
        'Precision': precision_score(y_true, predictions, zero_division=0),
        'Recall': recall_score(y_true, predictions, zero_division=0),
        'F1': f1_score(y_true, predictions, zero_division=0),
        'Balanced accuracy': balanced_accuracy_score(y_true, predictions),
    }
    if probabilities is not None:
        result['ROC AUC'] = roc_auc_score(y_true, probabilities)
    return result

## 4. Fit XGBoost and choose tree depth on validation data
XGBoost builds an ensemble of trees sequentially, optimizing the classification loss. Later trees improve the current ensemble's fit. It can model nonlinear relationships and interactions among sensor inputs.

Compare four maximum tree depths: 1, 2, 3 and 4. Smaller depth limits each tree's complexity; depth is not equivalent to LR's C. Keep 100 boosting rounds and learning rate 0.1 fixed, with L2 regularization 1.0, no class reweighting, and CPU execution. Select the highest **validation occupied-class F1**; ties favor shallower depth. The threshold stays fixed at 0.5.

Only training data fits the trees. Validation selects depth. There is no early stopping and no train-plus-validation refit, matching the earlier workflow's separate selection step. Four candidates provide a small, transparent search; this is not a claim of comprehensive tuning.

In [11]:
candidate_models = {}
validation_rows = []
for depth in [1, 2, 3, 4]:
    model = XGBClassifier(
        objective='binary:logistic',
        n_estimators=100,
        max_depth=depth,
        learning_rate=0.1,
        reg_lambda=1.0,
        min_child_weight=1,
        subsample=1.0,
        colsample_bytree=1.0,
        scale_pos_weight=1.0,
        tree_method='hist',
        device='cpu',
        n_jobs=2,
        eval_metric='logloss',
        random_state=SEED,
    )
    model.fit(X_train, y_train)
    val_prob = model.predict_proba(X_val)[:, 1]
    val_pred = (val_prob >= 0.5).astype(int)
    validation_rows.append({'Max depth': depth, **evaluate(y_val, val_pred, val_prob)})
    candidate_models[depth] = model

validation_results = pd.DataFrame(validation_rows).sort_values(
    ['F1', 'Max depth'], ascending=[False, True]).reset_index(drop=True)
best_depth = int(validation_results.loc[0, 'Max depth'])
best_model = candidate_models[best_depth]
print(validation_results.round(4).to_string(index=False))
print(f'Selected max_depth = {best_depth} using validation occupied-class F1.')

 Max depth  Accuracy  Precision  Recall     F1  Balanced accuracy  ROC AUC
         1    0.9918     0.9481  0.9905 0.9689             0.9913   0.9951
         2    0.9918     0.9481  0.9905 0.9689             0.9913   0.9947
         4    0.9319     0.6547  0.9919 0.7888             0.9575   0.9913
         3    0.9309     0.6529  0.9837 0.7849             0.9534   0.9811
Selected max_depth = 1 using validation occupied-class F1.


## 5. Evaluate the selected model on the test set
Keep the selected model trained on the training split; do not refit on validation data in this baseline. Evaluate the final setting on the test set after selection. Once test results are inspected, do not keep tuning against that set and call it an untouched final test.

In [12]:
test_prob = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)
test_results = pd.DataFrame([
    {'Model': f'XGBoost (max_depth={best_depth})', **evaluate(y_test, test_pred, test_prob)},
])
print(test_results.round(4).to_string(index=False))
print('Classification report:')
report = classification_report(y_test, test_pred, labels=[0, 1],
                               target_names=['Unoccupied', 'Occupied'],
                               digits=4, zero_division=0)
print('\n'.join(line for line in report.splitlines() if 'macro avg' not in line))
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
print('Confusion matrix (rows = actual, columns = predicted):')
print(pd.DataFrame(cm, index=['Actual unoccupied', 'Actual occupied'],
                   columns=['Predicted unoccupied', 'Predicted occupied']).to_string())
tn, fp, fn, tp = cm.ravel()
print(f'Correct unoccupied: {tn}; false occupancy alarms: {fp}; missed occupied: {fn}; correct occupied: {tp}.')

                Model  Accuracy  Precision  Recall     F1  Balanced accuracy  ROC AUC
XGBoost (max_depth=1)    0.9948     0.9865  0.9973 0.9918             0.9954   0.9985
Classification report:
              precision    recall  f1-score   support

  Unoccupied     0.9987    0.9936    0.9961      2343
    Occupied     0.9865    0.9973    0.9918      1097

    accuracy                         0.9948      3440
weighted avg     0.9948    0.9948    0.9948      3440
Confusion matrix (rows = actual, columns = predicted):
                   Predicted unoccupied  Predicted occupied
Actual unoccupied                  2328                  15
Actual occupied                       3                1094
Correct unoccupied: 2328; false occupancy alarms: 15; missed occupied: 3; correct occupied: 1094.


## 6. Inspect feature importance and the training/validation gap
Tree-based XGBoost does not have LR-style sensor coefficients. Display normalized **gain importance**: relative contribution of each feature to improvements in training loss across tree splits. These values are not signed effects or causal importance, and correlated predictors can share or distort importance.

Also report training and validation metrics for the selected model. A noticeably higher training score can indicate overfitting or differences between periods; a small gap is reassuring, not proof. This diagnostic does not retune the model. Test results must not guide later hyperparameter changes while still being described as untouched.

In [13]:
gain_scores = best_model.get_booster().get_score(importance_type='gain')
importance = np.array([gain_scores.get(feature, 0.0) for feature in FEATURES])
if importance.sum() > 0:
    importance = importance / importance.sum()
feature_importance = pd.DataFrame({
    'Sensor': FEATURES,
    'Normalized gain importance': importance,
}).sort_values('Normalized gain importance', ascending=False)
print(feature_importance.round(4).to_string(index=False))

train_prob = best_model.predict_proba(X_train)[:, 1]
val_prob = best_model.predict_proba(X_val)[:, 1]
fit_diagnostics = pd.DataFrame([
    {'Split': 'Training', **evaluate(y_train, (train_prob >= 0.5).astype(int), train_prob)},
    {'Split': 'Validation', **evaluate(y_val, (val_prob >= 0.5).astype(int), val_prob)},
])
print('Training versus validation:')
print(fit_diagnostics.round(4).to_string(index=False))
print(f"Training - validation F1 gap: {fit_diagnostics.loc[0, 'F1'] - fit_diagnostics.loc[1, 'F1']:.4f}")

     Sensor  Normalized gain importance
      Light                      0.9097
        CO2                      0.0479
   Humidity                      0.0252
Temperature                      0.0172
Training versus validation:
     Split  Accuracy  Precision  Recall     F1  Balanced accuracy  ROC AUC
  Training    0.9859     0.9518  0.9955 0.9732             0.9891   0.9978
Validation    0.9918     0.9481  0.9905 0.9689             0.9913   0.9951
Training - validation F1 gap: 0.0043


## 7. Review mistakes and explain your experiment
The first ten misclassified test records are shown below. Do not automatically change their labels based on the model: a prediction error does not establish a labeling error.

In [17]:
test_review = datasets['Test'][['date'] + FEATURES + [TARGET]].copy()
test_review['Predicted occupancy'] = test_pred
test_review['Probability occupied'] = test_prob
errors = test_review[test_review[TARGET] != test_review['Predicted occupancy']]
print(f'Misclassified test readings: {len(errors):,} / {len(test_review):,}')
print(errors.head(18).round(4).to_string(index=False))

Misclassified test readings: 18 / 3,440
               date  Temperature  Humidity  Light       CO2  Occupancy  Predicted occupancy  Probability occupied
2015-02-16 08:47:00      20.5000   25.0500 389.00  856.5000          0                    1                0.7147
2015-02-16 08:48:00      20.5000   25.0250 397.00  870.7500          0                    1                0.9139
2015-02-16 08:49:00      20.6000   25.0500 405.00  869.0000          0                    1                0.9376
2015-02-16 08:50:00      20.5500   25.1000 405.00  872.5000          0                    1                0.9376
2015-02-16 08:51:00      20.6000   25.2000 405.00  871.5000          0                    1                0.9376
2015-02-16 11:55:00      21.5000   28.1750 454.00  967.5000          0                    1                0.9851
2015-02-16 11:55:59      21.5250   28.1750 454.00  967.0000          0                    1                0.9851
2015-02-16 11:57:00      21.5000   28.1500 454.0